# M8_P1 - scRNA-Seq Analysis in Seurat

In this practice, we cluster and annotate single-cell transcriptomes from the *Arabidopsis* root, using the wild-type data of [Ryu *et al.* (2019)](https://doi.org/10.1104/pp.18.01482) ([GSE123013](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE123013)). We follow the processing steps described in the paper's Materials and Methods, translated into the current version of [Seurat](https://satijalab.org/seurat/) (v5). In M8_L2 you made a count matrix for part of one sample with STARsolo. Here we start from the authors' own Cell Ranger matrices for all three wild-type replicates (7,522 cells), so that our results can be compared with the paper.

**NOTE:** This notebook uses the R kernel from the `module_8_seurat` conda environment (`environment/module_8_seurat.yml`). It needs about 8 GB of RAM; on Hellbender, request at least 2 cores and 16 GB for the Jupyter session.

<u>**By the end of this notebook, you should be able to:**</u>
- Load 10x count matrices into Seurat and apply the gene and cell filters used by Ryu *et al.*
- Calculate and plot per-cell QC metrics, including mitochondrial and chloroplast fractions
- Normalize the data, select 1,000 highly variable genes and run PCA
- Integrate three replicates with CCA, then cluster the cells and visualize them with t-SNE and UMAP
- Find marker genes for each cluster, and name the clusters using known root cell-type markers
- Compare your clustering with the published one, and explain why they differ

```{admonition} Adapted from
:class: tip
**Source:** The order of the analysis and the Seurat function calls follow the Satija Lab vignettes [*Seurat - Guided Clustering Tutorial*](https://satijalab.org/seurat/articles/pbmc3k_tutorial) and [*Integrative analysis in Seurat v5*](https://satijalab.org/seurat/articles/seurat5_integration) (Seurat is released under the [MIT License](https://github.com/satijalab/seurat/blob/master/LICENSE.md)). The reasoning behind the QC metrics follows the "Quality control" lessons of the Harvard Chan Bioinformatics Core course [*Introduction to single-cell RNA-seq*](https://hbctraining.github.io/Intro-to-scRNAseq/) (Piper *et al.* 2022, [doi:10.5281/zenodo.5826256](https://doi.org/10.5281/zenodo.5826256), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)). The idea of annotating root clusters by marker-gene panels follows the Galaxy Training Network tutorial [*Analysis of plant scRNA-Seq Data with Scanpy*](https://training.galaxyproject.org/training-material/topics/single-cell/tutorials/scrna-plant/tutorial.html) (Tekman, Serrano-Solano, Gallardo, Videm; CC BY 4.0).

**Changes:** The human PBMC and pancreas examples of the vignettes were replaced by *Arabidopsis* root protoplasts (GSE123013). The filters, the number of variable genes, the integration dimensions and the clustering resolution were set to the values in Ryu *et al.* (2019). Plant-specific QC (chloroplast genes), AGI gene IDs, the module-score annotation, the mutant comparison and all text and questions were written for this course. No text or figures were copied.
```

## 1. What did the paper do?

The Materials and Methods of Ryu *et al.* (2019) describe the analysis in one paragraph. The table below lists each step next to the Seurat v5 equivalent used in this notebook. The authors used Seurat 2.3.3 (2018); several functions have been renamed or replaced since then.

| Step | Ryu *et al.* (2019) | This notebook (Seurat v5) |
|---|---|---|
| Alignment and counting | Cell Ranger 2.1.1, TAIR10 reference, default settings | Same matrices, downloaded from GEO |
| Gene filter | Genes expressed in more than 3 cells | `CreateSeuratObject(min.cells = 4)` |
| Cell filter | At least 200 expressed genes per cell | `CreateSeuratObject(min.features = 200)` |
| Normalization | Log-normalized, scaled | `NormalizeData()` (LogNormalize), `ScaleData()` |
| Batch correction | ComBat (sva package), then Seurat multi-CCA on the union of the top 1,000 variable genes of each dataset | `FindVariableFeatures(nfeatures = 1000)` per replicate, then `IntegrateLayers(method = CCAIntegration)` |
| Dimensions | Top 20 aligned CCs | `dims = 1:20` |
| Clustering | `FindClusters`, resolution 0.2 (three wild-type replicates) | `FindNeighbors()` + `FindClusters(resolution = 0.2)` |
| Visualization | t-SNE (perplexity 30, seed 1) | `RunTSNE(perplexity = 30, seed.use = 1)`, and UMAP |
| Markers | `FindConservedMarkers`, average fold change ≥ 1 and adjusted *P* < 0.05 | `FindConservedMarkers()` and `FindAllMarkers()` |
| Annotation | Expression of 86 known root cell-type markers (their Supplemental Table S3) | A smaller panel of well-known markers, scored with `AddModuleScore()` |

```{note}
**ComBat** was used in 2018 because the CCA alignment in Seurat 2 worked best on data already corrected for simple shifts between batches. In Seurat v5, `IntegrateLayers()` with `CCAIntegration` finds "anchors" (pairs of similar cells in different replicates) and corrects the data itself, so ComBat is no longer part of the standard workflow. This is one reason our results will not be identical to the paper's.
```

## 2. Setup

### Load libraries

In [ ]:
# If library(Seurat) fails with an error about a package in ~/.R/lib, your personal R library
# (set by R_LIBS_USER in ~/.Renviron) is overriding the conda environment.
# Start Jupyter with R_ENVIRON_USER=/dev/null, or remove that line from ~/.Renviron.
library(Seurat)
library(tidyverse)
library(patchwork)

set.seed(1)                       # the paper used random seed 1
options(repr.plot.width = 12, repr.plot.height = 6)
packageVersion("Seurat")

### Get the data
Download the Cell Ranger matrices from GEO with `bash data/fetch_data.sh scrna_ryu` (about 190 MB). The script extracts one folder per sample into `data/scrna/GSE123013/`:

```
data/scrna/GSE123013/
├── Sample_WT-WERGFP/          # WT_rep1, GSM3490689
│   ├── filtered_gene_bc_matrices/TAIR10/
│   │   ├── barcodes.tsv       # one cell barcode per line (the columns of the matrix)
│   │   ├── genes.tsv          # AGI ID and gene symbol (the rows)
│   │   └── matrix.mtx         # sparse matrix of UMI counts
│   └── web_summary.html       # the Cell Ranger QC report for this sample
├── Sample_WT-WERGFP_2/        # WT_rep2, GSM3490690 (the run you aligned in M8_L2)
├── Sample_WT-WERGFP_3/        # WT_rep3, GSM3490691
├── Sample_rhd6/               # rhd6-1, GSM3490692
└── Sample_gl2/                # gl2, GSM3490693
```

These are Cell Ranger 2 `filtered_gene_bc_matrices`: only the barcodes that Cell Ranger called as cells. Open one of the `web_summary.html` files in a browser and compare it with the STARsolo `Summary.csv` from M8_L2.

In [ ]:
DATA_DIR <- Sys.getenv("DATA_DIR", unset = "../../../data")   # repository data/ folder
mtx_dir  <- file.path(DATA_DIR, "scrna", "GSE123013")
list.files(mtx_dir)

# The three wild-type replicates (all carry the WER::GFP reporter in a Col-0 background)
wt_samples <- c(WT_rep1 = "Sample_WT-WERGFP",
                WT_rep2 = "Sample_WT-WERGFP_2",
                WT_rep3 = "Sample_WT-WERGFP_3")

### Gene IDs and gene symbols
`genes.tsv` has two columns, the AGI locus ID (for example `AT1G79840`) and a gene symbol (`GL2`). By default, `Read10X()` uses the symbols as row names. For *Arabidopsis* this is risky: many genes have no symbol, and some symbols are shared by more than one locus. We will therefore use the **AGI IDs** as row names and keep a lookup table to translate them into symbols for plots.

In [ ]:
genes <- read.delim(file.path(mtx_dir, wt_samples[[1]], "filtered_gene_bc_matrices", "TAIR10", "genes.tsv"),
                    header = FALSE, col.names = c("agi", "symbol"))
nrow(genes)

# Symbols that appear for more than one AGI ID
dup <- genes %>% group_by(symbol) %>% filter(n() > 1) %>% arrange(symbol)
length(unique(dup$symbol))
dup %>% filter(symbol %in% c("SMB", "BRN1", "PLT1", "PLT2"))

# A helper to turn AGI IDs into "SYMBOL (AGI)" labels for plots
agi2label <- function(ids) {
  sym <- genes$symbol[match(ids, genes$agi)]
  ifelse(is.na(sym) | sym == ids, ids, paste0(sym, " (", ids, ")"))
}

## 3. Create Seurat objects and apply the paper's filters

`CreateSeuratObject()` can filter while it builds the object:
- `min.cells = 4` keeps genes detected in **more than 3 cells**
- `min.features = 200` keeps cells with **at least 200 detected genes**

We make one object per replicate, as the authors filtered each dataset separately.

In [ ]:
wt_list <- lapply(names(wt_samples), function(s) {
  counts <- Read10X(file.path(mtx_dir, wt_samples[[s]], "filtered_gene_bc_matrices", "TAIR10"),
                    gene.column = 1)          # column 1 = AGI IDs
  CreateSeuratObject(counts = counts, project = s, min.cells = 4, min.features = 200)
})
names(wt_list) <- names(wt_samples)

# cells and genes kept per replicate
sapply(wt_list, ncol)
sapply(wt_list, nrow)

The paper reports 7,522 wild-type cells in total, which should match the sum above. No cells are lost to the 200-gene filter, because Cell Ranger had already removed barcodes with little RNA.

Next, we merge the replicates into one object. In Seurat v5, each replicate stays in its own **layer** (`counts.WT_rep1`, `counts.WT_rep2`, ...), which is what the integration step needs later. `add.cell.ids` puts the replicate name in front of each barcode, so identical barcodes from different runs cannot be confused.

In [ ]:
root <- merge(wt_list[[1]], y = wt_list[-1], add.cell.ids = names(wt_list))
root$replicate <- root$orig.ident
root
rm(wt_list); invisible(gc())

## 4. Quality control

For every cell, Seurat has already calculated:
- `nCount_RNA`: the total number of UMIs (molecules) detected
- `nFeature_RNA`: the number of genes with at least one UMI

We add the fraction of UMIs from the two organellar genomes. In TAIR10, mitochondrial genes start with `ATMG` and chloroplast genes with `ATCG`. In animal cells, a high mitochondrial fraction usually means a broken cell that has lost its cytoplasmic mRNA. In plant protoplasts, it is a similar warning sign, and a high chloroplast fraction in a root sample can point to contamination from green tissue.

In [ ]:
root[["percent.mt"]] <- PercentageFeatureSet(root, pattern = "^ATMG")
root[["percent.cp"]] <- PercentageFeatureSet(root, pattern = "^ATCG")

# median QC metrics per replicate
root@meta.data %>%
  group_by(replicate) %>%
  summarise(cells = n(),
            median_UMIs = median(nCount_RNA),
            median_genes = median(nFeature_RNA),
            median_pct_mt = median(percent.mt),
            max_pct_mt = max(percent.mt),
            max_pct_cp = max(percent.cp))

In [ ]:
VlnPlot(root, features = c("nCount_RNA", "nFeature_RNA", "percent.mt", "percent.cp"),
        group.by = "replicate", ncol = 4, pt.size = 0.1)

In [ ]:
p1 <- FeatureScatter(root, feature1 = "nCount_RNA", feature2 = "nFeature_RNA", group.by = "replicate")
p2 <- FeatureScatter(root, feature1 = "nCount_RNA", feature2 = "percent.mt", group.by = "replicate")
p1 + p2

**How to read these plots:**
- The paper reports a median of about 24,000 UMIs and 5,000 genes per cell. That is far more than a typical animal cell, because the authors sequenced deeply (about 75,000 reads per cell).
- Genes and UMIs should rise together (left plot). Cells far above the main cloud, with many more UMIs and genes than the rest, may be **doublets**.
- The organellar fractions here are very low (well under 1% for almost every cell), so there is no sign of a large population of broken cells.

Ryu *et al.* applied **no filters beyond the 200-gene minimum**. To reproduce their analysis, we keep all cells. In your own data you would usually also remove cells with very high gene counts (possible doublets) and a high organellar fraction, or use a dedicated doublet detector such as [scDblFinder](https://bioconductor.org/packages/scDblFinder/). Exercise 1 asks you to test what a stricter filter changes.

## 5. Normalization, variable genes and PCA

1. **`NormalizeData()`** divides each cell's counts by its total UMIs, multiplies by 10,000 and takes `log1p`. This removes most of the effect of sequencing depth.
2. **`FindVariableFeatures()`** finds the genes whose variance across cells is higher than expected for their mean (the `vst` method). Because the replicates are in separate layers, this runs on each replicate, and Seurat combines the results, much like the paper's union of top genes per dataset. We keep 1,000 genes, as the authors did.
3. **`ScaleData()`** centers each variable gene to mean 0 and variance 1, so that highly expressed genes do not dominate the PCA.
4. **`RunPCA()`** summarizes the 1,000 genes as principal components.

In [ ]:
root <- NormalizeData(root, normalization.method = "LogNormalize", scale.factor = 10000)
root <- FindVariableFeatures(root, selection.method = "vst", nfeatures = 1000)

# label the 15 most variable genes with their symbols
top15 <- head(VariableFeatures(root), 15)
LabelPoints(plot = VariableFeaturePlot(root), points = top15, labels = agi2label(top15), repel = TRUE)

In [ ]:
root <- ScaleData(root)
root <- RunPCA(root, npcs = 30, verbose = FALSE)
ElbowPlot(root, ndims = 30)

The elbow plot shows how much variation each principal component explains. Here it flattens out at around 15 to 20 PCs, which is consistent with the paper's choice of 20 dimensions.

Before integrating, it is worth checking how strong the replicate (batch) effect is. We cluster and embed the **unintegrated** PCA and color the cells by replicate.

In [ ]:
root <- RunUMAP(root, reduction = "pca", dims = 1:20, reduction.name = "umap.unintegrated", verbose = FALSE)
DimPlot(root, reduction = "umap.unintegrated", group.by = "replicate") + ggtitle("Before integration")

## 6. Integration of the three replicates (CCA)

`IntegrateLayers()` with `CCAIntegration`:
1. Projects pairs of replicates into a shared space with canonical correlation analysis (CCA)
2. Finds **anchors**, which are pairs of mutually nearest cells in two replicates
3. Uses the anchors to correct the cells' coordinates, producing a new reduction (`integrated.cca`) in which replicates overlap

All downstream steps (neighbors, clusters, t-SNE, UMAP) then use `integrated.cca` instead of `pca`.

In [ ]:
root <- IntegrateLayers(root, method = CCAIntegration,
                        orig.reduction = "pca", new.reduction = "integrated.cca",
                        dims = 1:20, verbose = FALSE)

## 7. Clustering, t-SNE and UMAP

Seurat clusters cells on a graph: `FindNeighbors()` connects each cell to its most similar cells, and `FindClusters()` cuts the graph into communities (Louvain algorithm). The **resolution** controls how many clusters you get: higher values give more and smaller clusters. The paper used 0.2 for the three wild-type replicates.

In [ ]:
root <- FindNeighbors(root, reduction = "integrated.cca", dims = 1:20)
root <- FindClusters(root, resolution = 0.2, random.seed = 1)

# the same seed and perplexity as the paper
root <- RunTSNE(root, reduction = "integrated.cca", dims = 1:20, perplexity = 30, seed.use = 1)
root <- RunUMAP(root, reduction = "integrated.cca", dims = 1:20, seed.use = 1, verbose = FALSE)

table(Cluster = root$seurat_clusters, Replicate = root$replicate)

In [ ]:
p_rep  <- DimPlot(root, reduction = "tsne", group.by = "replicate") + ggtitle("t-SNE: replicate")
p_clus <- DimPlot(root, reduction = "tsne", label = TRUE) + ggtitle("t-SNE: clusters (res 0.2)") + NoLegend()
p_umap <- DimPlot(root, reduction = "umap", label = TRUE) + ggtitle("UMAP: clusters") + NoLegend()
p_rep + p_clus + p_umap

**How many clusters did you get?** Ryu *et al.* found **nine** major clusters at resolution 0.2. With Seurat v5 you will probably get more (13 in our run with the settings above). That does not mean either answer is wrong:
- The integration method has changed (ComBat + Seurat 2 multi-CCA, compared with Seurat v5 anchor-based CCA), and so has the neighbor graph that the clustering runs on.
- A resolution value is not comparable between software versions. It is a tuning knob, not a biological constant.

What matters is whether the clusters correspond to cell types. A small cluster in our analysis may be a sub-population that the paper merged into a larger cluster (or the reverse). In Exercise 2 you will try other resolutions and see which clusters split or merge.

The t-SNE colored by replicate should show the three replicates mixed together in every cluster. Compare it with the unintegrated UMAP above.

## 8. Marker genes

First we join the replicate layers back into one counts and one data layer, which the differential expression functions need.

- **`FindAllMarkers()`** compares each cluster with all other cells (Wilcoxon test).
- **`FindConservedMarkers()`** runs this comparison **separately in each replicate**, and keeps genes that are markers in all of them. This is the test the paper used. It is stricter, because a marker driven by one replicate is rejected.

We use the paper's thresholds: an average log2 fold change of at least 1, and an adjusted *P*-value below 0.05.

In [ ]:
root <- JoinLayers(root)

all_markers <- FindAllMarkers(root, only.pos = TRUE, logfc.threshold = 1, min.pct = 0.25, verbose = FALSE)
all_markers <- all_markers %>% filter(p_val_adj < 0.05) %>% mutate(label = agi2label(gene))

# number of markers per cluster, and the top 5 per cluster
table(all_markers$cluster)
all_markers %>% group_by(cluster) %>% slice_max(avg_log2FC, n = 5) %>%
  select(cluster, label, avg_log2FC, pct.1, pct.2, p_val_adj) %>% as.data.frame()

In [ ]:
# Conserved markers for one cluster (this test is slow, so we run it for cluster 0 only).
# The columns are prefixed with the replicate name; max_pval and minimump_p_val combine the replicates.
cons0 <- FindConservedMarkers(root, ident.1 = "0", grouping.var = "replicate",
                              only.pos = TRUE, logfc.threshold = 1, verbose = FALSE)
cons0$label <- agi2label(rownames(cons0))
head(cons0[, c("label", "WT_rep1_avg_log2FC", "WT_rep2_avg_log2FC", "WT_rep3_avg_log2FC", "max_pval")], 10)

To find out what a cluster is, look up its top markers in [TAIR](https://www.arabidopsis.org/) or in a root expression atlas such as the [Arabidopsis eFP Browser](https://bar.utoronto.ca/efp/cgi-bin/efpWeb.cgi). For example, if a cluster's markers include cell-wall and root-hair genes, it is likely to be trichoblasts.

## 9. Annotating clusters with known marker genes

The paper assigned clusters with 86 marker genes from earlier studies (their Supplemental Table S3). We use a smaller panel of well-studied markers. Each gene below has been checked in the matrix's `genes.tsv`, and the IDs are AGI loci, not symbols.

| Cell type | Marker genes (AGI) | Notes |
|---|---|---|
| Root hair (trichoblast) | COBL9 (AT5G49270), EXPA7 (AT1G12560), PRP3 (AT3G62680), MRH6 (AT2G03720) | Cell-wall genes needed for hair growth |
| Non-hair (atrichoblast) | GL2 (AT1G79840), WER (AT5G14750), CPC (AT2G46410) | WER and GL2 specify non-hair fate; CPC is transcribed in non-hair cells |
| Cortex | CYP71B34 (AT3G26300), IAMT1 (AT5G55250) | Cortex markers used by Denyer *et al.* (2019) and the Galaxy plant tutorial |
| Endodermis | SCR (AT3G54220), MYB36 (AT5G57620), CASP1 (AT2G36100) | MYB36 and CASP1 mark Casparian strip formation |
| Stele (vascular tissue) | SHR (AT4G37650), ATHB-8 (AT4G32880), APL (AT1G79430), SUC2 (AT1G22710) | SHR: stele; ATHB-8: xylem; APL and SUC2: phloem |
| Root cap / columella | SMB (AT1G79580), FEZ (AT1G26870), BRN1 (AT1G33280) | Lateral root cap and columella |
| Dividing (meristematic) cells | CYCB1;1 (AT4G37490), CYCB1;2 (AT5G06150) | Expressed in G2/M of the cell cycle |
| Quiescent center / stem cell niche | WOX5 (AT3G11260), AGL42 (AT5G62165), PLT1 (AT3G20840) | Very rare cells; used for feature plots only |

We left out one classic xylem marker, AHP6 (AT1G80100). It is in `genes.tsv`, but only one wild-type cell has a UMI for it, so the `min.cells = 4` filter removes it. A marker that works well in reporter lines or bulk RNA-Seq can be too weakly expressed to be detected in single cells.

`DotPlot()` shows, for each cluster, the fraction of cells expressing a gene (dot size) and its average expression (color).

In [ ]:
marker_sets <- list(
  "Root hair"     = c("AT5G49270", "AT1G12560", "AT3G62680", "AT2G03720"),
  "Non-hair"      = c("AT1G79840", "AT5G14750", "AT2G46410"),
  "Cortex"        = c("AT3G26300", "AT5G55250"),
  "Endodermis"    = c("AT3G54220", "AT5G57620", "AT2G36100"),
  "Stele"         = c("AT4G37650", "AT4G32880", "AT1G79430", "AT1G22710"),
  "Root cap"      = c("AT1G79580", "AT1G26870", "AT1G33280"),
  "Dividing"      = c("AT4G37490", "AT5G06150")
)
qc_niche <- c("AT3G11260", "AT5G62165", "AT3G20840")

# check that every marker survived the gene filter
setdiff(c(unlist(marker_sets), qc_niche), rownames(root))

panel <- unique(unlist(marker_sets))
options(repr.plot.width = 16, repr.plot.height = 6)
DotPlot(root, features = panel) +
  scale_x_discrete(labels = agi2label) +
  RotatedAxis() + ggtitle("Known root markers by cluster")

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 10)
show <- c("AT5G49270", "AT1G79840", "AT3G26300", "AT3G54220", "AT4G37650", "AT1G79580", "AT4G37490", "AT3G11260")
plots <- lapply(show, function(g) FeaturePlot(root, features = g, reduction = "tsne", order = TRUE) + ggtitle(agi2label(g)))
wrap_plots(plots, ncol = 4)

### Scoring clusters with marker panels
Reading a dot plot by eye works, but it is easy to be inconsistent. `AddModuleScore()` gives every cell a score for each marker set: the average expression of the set minus that of a random set of control genes with similar expression levels. We then take the mean score of each cluster and give the cluster the name of its highest-scoring cell type.

```{note}
This is a quick, transparent first pass, not a final answer. Always check it against the dot plot and the cluster's own top markers. A cluster whose best score is low, or close to its second-best score, needs a closer look; it may be a transition state (for example meristematic cells that are just starting to differentiate) or a mixture.
```

In [ ]:
root <- AddModuleScore(root, features = marker_sets, name = "score_", seed = 1)
score_cols <- paste0("score_", seq_along(marker_sets))
colnames(root@meta.data)[match(score_cols, colnames(root@meta.data))] <- make.names(names(marker_sets))

cluster_scores <- root@meta.data %>%
  group_by(cluster = seurat_clusters) %>%
  summarise(cells = n(), across(all_of(make.names(names(marker_sets))), mean))

assignment <- cluster_scores %>%
  pivot_longer(-c(cluster, cells), names_to = "cell_type", values_to = "score") %>%
  group_by(cluster, cells) %>%
  arrange(desc(score), .by_group = TRUE) %>%
  summarise(cell_type = first(cell_type), best = first(score), second = nth(score, 2), .groups = "drop") %>%
  mutate(cell_type = gsub("\\.", " ", cell_type), margin = best - second)
assignment

In [ ]:
root$cell_type <- assignment$cell_type[match(root$seurat_clusters, assignment$cluster)]

options(repr.plot.width = 14, repr.plot.height = 6)
DimPlot(root, reduction = "tsne", group.by = "cell_type", label = TRUE, repel = TRUE) + ggtitle("t-SNE: assigned cell types") |
  DimPlot(root, reduction = "umap", group.by = "cell_type", label = TRUE, repel = TRUE) + ggtitle("UMAP: assigned cell types")

# proportion of each cell type in each replicate
round(prop.table(table(root$cell_type, root$replicate), margin = 2) * 100, 1)

**Check the `best` column.** A module score is relative to random control genes, so a score at or below zero means the cluster expresses that panel no more than background. In our run, clusters 0 and 12 had negative best scores: no panel fits them, and their "Dividing" label only means it was the least negative. Treat such clusters as unassigned until you have looked at their own markers. For example, the top markers of our cluster 12 (LAC17, LAC11, MYB85) are lignin genes, which points to differentiating xylem, a cell type our small panel does not cover.

Several clusters may share a label (for example two root-hair clusters). This is expected: in the root, the same cell type is present at several developmental stages, and the clustering often separates young cells near the meristem from mature cells. Ryu *et al.* used exactly this structure, in a second analysis of the epidermal clusters, to order root-hair and non-hair cells along a developmental pseudotime.

Save the annotated object so you can come back to it without re-running everything:

In [ ]:
dir.create("results", showWarnings = FALSE)
saveRDS(root, file = "results/M8_P1_wt_root_annotated.rds")

## 10. Optional: wild type compared with the *rhd6* and *gl2* mutants

The second analysis in the paper merged the wild-type cells with the two epidermal mutants and clustered them at resolution 0.3:
- ***rhd6-1*** (ROOT HAIR DEFECTIVE 6) lacks root hairs.
- ***gl2*** (GLABRA2) makes hairs from cells that would normally become non-hair cells.

If the clusters are right, the mutants should change the **proportions** of the epidermal clusters. The code below repeats the workflow with all five samples. It takes several minutes and about 10 GB of RAM.

In [ ]:
all_samples <- c(wt_samples, rhd6 = "Sample_rhd6", gl2 = "Sample_gl2")
five <- lapply(names(all_samples), function(s) {
  m <- Read10X(file.path(mtx_dir, all_samples[[s]], "filtered_gene_bc_matrices", "TAIR10"), gene.column = 1)
  CreateSeuratObject(m, project = s, min.cells = 4, min.features = 200)
})
five <- merge(five[[1]], y = five[-1], add.cell.ids = names(all_samples))
five$genotype <- ifelse(grepl("^WT", five$orig.ident), "WT", five$orig.ident)

five <- five %>% NormalizeData() %>% FindVariableFeatures(nfeatures = 1000) %>% ScaleData() %>% RunPCA(verbose = FALSE)
five <- IntegrateLayers(five, method = CCAIntegration, orig.reduction = "pca",
                        new.reduction = "integrated.cca", dims = 1:20, verbose = FALSE)
five <- FindNeighbors(five, reduction = "integrated.cca", dims = 1:20)
five <- FindClusters(five, resolution = 0.3, random.seed = 1)
five <- RunTSNE(five, reduction = "integrated.cca", dims = 1:20, seed.use = 1)
five <- JoinLayers(five)

five <- AddModuleScore(five, features = marker_sets[c("Root hair", "Non-hair")], name = "epi_", seed = 1)
DimPlot(five, reduction = "tsne", split.by = "genotype", label = TRUE) + NoLegend()

# percentage of each genotype's cells in each cluster, plus the mean hair and non-hair scores of the cluster
pct <- round(prop.table(table(five$seurat_clusters, five$genotype), margin = 2) * 100, 1)
epi <- five@meta.data %>% group_by(seurat_clusters) %>% summarise(hair_score = mean(epi_1), nonhair_score = mean(epi_2))
cbind(as.data.frame.matrix(pct), epi[, -1])

## Questions:
Answer the questions below in the text box at the end of this page. Use the code cells for the exercises.

1. **Exercise 1 (QC):** Animal workflows often remove cells with more than 5% (the Seurat PBMC tutorial) or 20% (the HBC course) mitochondrial UMIs. Look at `percent.mt` in the QC table and violin plot: would such a cut-off remove any cells here? Why might mitochondrial reads be so rare in these root protoplasts? Then remove cells with more than 10,000 detected genes and re-run the workflow. How many cells are removed from each replicate? Do any clusters disappear or shrink sharply? What might those cells have been?
2. **Exercise 2 (resolution):** Re-run `FindClusters()` at resolutions 0.1, 0.3 and 0.5. Make a table of the number of clusters at each resolution. Which cell types split first as the resolution increases? Which resolution gives a number of clusters closest to the paper's nine, and do those clusters match the paper's cell types?
3. **Exercise 3 (markers):** Pick one cluster that the module score left uncertain (a small `margin`). Look up its top five markers from `FindAllMarkers()` in TAIR. What cell type or state do you think it is?
4. In the integration step, what would go wrong if one replicate contained a cell type that was missing from the other two? (Hint: think about what anchors are.)
5. The paper reports a median of about 5,000 genes per cell, while many animal datasets have 1,000 to 2,000. Give two reasons why these root protoplasts might show so many genes per cell.
6. **(Optional section 10)** Which clusters are depleted in *rhd6* and which are expanded in *gl2*? Are the changes consistent with the mutants' phenotypes?

In [ ]:
# Exercise 1: your code here

In [ ]:
# Exercise 2: your code here

In [ ]:
# Exercise 3: your code here

In [ ]:
sessionInfo()

## References / Resources
- Ryu KH, Huang L, Kang HM, Schiefelbein J (2019). Single-cell RNA sequencing resolves molecular relationships among individual plant cells. *Plant Physiology* 179:1444-1456. [doi:10.1104/pp.18.01482](https://doi.org/10.1104/pp.18.01482). Data: [GSE123013](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE123013)
- Hao Y, Stuart T, Kowalski MH, *et al.* (2024). Dictionary learning for integrative, multimodal and scalable single-cell analysis. *Nature Biotechnology* 42:293-304. [doi:10.1038/s41587-023-01767-y](https://doi.org/10.1038/s41587-023-01767-y) (Seurat v5)
- Butler A, Hoffman P, Smibert P, Papalexi E, Satija R (2018). Integrating single-cell transcriptomic data across different conditions, technologies, and species. *Nature Biotechnology* 36:411-420. [doi:10.1038/nbt.4096](https://doi.org/10.1038/nbt.4096) (Seurat 2 and CCA, used by Ryu *et al.*)
- Denyer T, Ma X, Klesen S, *et al.* (2019). Spatiotemporal developmental trajectories in the *Arabidopsis* root revealed using high-throughput single-cell RNA sequencing. *Developmental Cell* 48:840-852. [doi:10.1016/j.devcel.2019.02.022](https://doi.org/10.1016/j.devcel.2019.02.022)
- Satija Lab: [Seurat - Guided Clustering Tutorial](https://satijalab.org/seurat/articles/pbmc3k_tutorial) and [Integrative analysis in Seurat v5](https://satijalab.org/seurat/articles/seurat5_integration)
- HBC training: [Introduction to single-cell RNA-seq](https://hbctraining.github.io/Intro-to-scRNAseq/) (CC BY 4.0)
- Galaxy Training Network: [Analysis of plant scRNA-Seq Data with Scanpy](https://training.galaxyproject.org/training-material/topics/single-cell/tutorials/scrna-plant/tutorial.html) (CC BY 4.0), a Python/Scanpy version of a root scRNA-Seq analysis on GSE123818
- For trajectories (pseudotime), see [Monocle 3](https://cole-trapnell-lab.github.io/monocle3/) and [Slingshot](https://bioconductor.org/packages/slingshot/)